# GNN sur un dataset massif : Tox21

**Nom :** ............................................................  
**Prénom :** ........................................................  

Puisque `ClinTox` était trop petit (1500 molécules) pour que le GNN devienne vraiment intelligent, nous passons à la vitesse supérieure !

Nous allons utiliser **Tox21** (Toxicology in the 21st Century) : une initiative majeure du gouvernement américain.
Ce dataset contient plus de **8 000 molécules**. 
Mieux encore : au lieu de prédire un simple "Toxique / Non Toxique", il prédit **12 cibles biologiques différentes** en même temps (ex: Est-ce que ça perturbe les hormones ? Est-ce que ça abîme l'ADN ?).

In [7]:
import logging
logging.getLogger("deepchem").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') 

import torch
import deepchem as dc
import numpy as np

In [8]:
# 1. FEATURIZER
featurizer = dc.feat.ConvMolFeaturizer()

# 2. CHARGEMENT DE TOX21 (Au lieu de ClinTox)
print("Téléchargement et préparation de Tox21 (plus de 8000 graphes, cela prend une bonne minute...)...")
import os
local_cache = "./datasets/tox21_cache"
os.makedirs(local_cache, exist_ok=True)
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer, save_dir=local_cache, data_dir=local_cache)
train_dataset, valid_dataset, test_dataset = datasets

print(f"\nTaille Entraînement: {len(train_dataset)} molécules")
print(f"Taille Test: {len(test_dataset)} molécules")
print("\nLes 12 types de toxicité étudiés (Récepteurs nucléaires, Stress cellulaire...) :\n", tasks)

Téléchargement et préparation de Tox21 (plus de 8000 graphes, cela prend une bonne minute...)...

Taille Entraînement: 6258 molécules
Taille Test: 783 molécules

Les 12 types de toxicité étudiés (Récepteurs nucléaires, Stress cellulaire...) :
 ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


## Partie 1 : Du binaire global au multi-tâches précis

### Questions :
1. **Avantage de Tox21** : Dans les TP précédents (ClinTox), nous essayions de prédire si une molécule était globalement « Toxique (1) » ou « Saine (0) ». Quel est l'avantage majeur du jeu de données **Tox21** par rapport à ClinTox pour un chimiste ou un pharmacologue ?
2. **Nombre de cibles** : Combien de tâches (tasks) biologiques différentes le modèle essaie-t-il de prédire simultanément dans ce notebook ?

*(Double-cliquez sur cette cellule pour rédiger vos réponses ci-dessous :)*

* **Réponse 1 :** ...
* **Réponse 2 :** ...


In [9]:
# 3. CRÉATION DU MODÈLE
# Le réseau aura 12 sorties indépendantes au lieu de 2 !
model = dc.models.GraphConvModel(
    n_tasks=len(tasks), 
    mode='classification',
    dropout=0.2
)

# 4. ENTRAÎNEMENT
# Attention, comme le dataset est 5 fois plus gros, chaque epoch sera plus longue.
print("Début de l'entraînement massif...")
model.fit(train_dataset, nb_epoch=10)
print("Entraînement terminé !")

Début de l'entraînement massif...
Entraînement terminé !


## Partie 2 : Mécanismes biologiques de toxicité

### Questions :
3. **Cibles NR vs SR** : Les 12 cibles biologiques de Tox21 sont divisées en deux grandes catégories : les **Récepteurs Nucléaires (NR)** et les **Réponses au Stress Cellulaire (SR)**.
   * Qu'est-ce qu'un perturbateur endocrinien (lié aux récepteurs comme `NR-AR` ou `NR-ER`) ?
   * Que signifie l'activation de la protéine **p53** (cible `SR-p53`) pour la cellule et l'intégrité de son ADN ?

*(Double-cliquez sur cette cellule pour rédiger vos réponses ci-dessous :)*

* **Réponse 3.1 (Perturbateurs endocriniens) :** ...
* **Réponse 3.2 (Protéine p53) :** ...


In [10]:
# 5. ÉVALUATION
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
train_scores = model.evaluate(train_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

print(f"Score moyen sur Entraînement : {train_scores['roc_auc_score']:.3f}")
print(f"Score moyen sur Test : {test_scores['roc_auc_score']:.3f}")

Score moyen sur Entraînement : 0.850
Score moyen sur Test : 0.685


In [11]:
# 6. TESTONS SUR LE CYANURE D'HYDROGÈNE
molecule = "C#N"
graphe_test = featurizer.featurize([molecule])
dataset_test = dc.data.NumpyDataset(X=graphe_test)

# La prédiction renvoie un tableau avec les 12 probabilités (une pour chaque cible biologique)
predictions = model.predict(dataset_test)[0]

print(f"\n--- PROFIL DE TOXICITÉ : {molecule} ---")
for index, nom_cible in enumerate(tasks):
    # L'indice [1] correspond à la probabilité d'être Toxique pour cette cible
    probabilite = predictions[index][1] 
    
    # On affiche une alerte si la probabilité dépasse 50%
    if probabilite > 0.5:
        print(f"️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
    else:
        print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")


--- PROFIL DE TOXICITÉ : C#N ---
  Sûr    sur NR-AR      : 33.8%
  Sûr    sur NR-AR-LBD  : 19.5%
  Sûr    sur NR-AhR     : 8.5%
  Sûr    sur NR-Aromatase : 5.8%
  Sûr    sur NR-ER      : 25.4%
  Sûr    sur NR-ER-LBD  : 18.9%
  Sûr    sur NR-PPAR-gamma : 9.7%
  Sûr    sur SR-ARE     : 24.1%
  Sûr    sur SR-ATAD5   : 37.5%
  Sûr    sur SR-HSE     : 43.0%
  Sûr    sur SR-MMP     : 5.2%
  Sûr    sur SR-p53     : 16.5%


In [12]:
# 7. TESTONS LE PARACÉTAMOL ET LE VALDÉCOXIB
smiles_a_tester = {
    "Paracétamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Valdécoxib": "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N"
}

noms = list(smiles_a_tester.keys())
smiles = list(smiles_a_tester.values())

graphes_medocs = featurizer.featurize(smiles)
dataset_medocs = dc.data.NumpyDataset(X=graphes_medocs)

predictions_medocs = model.predict(dataset_medocs)

for i, nom_molecule in enumerate(noms):
    print(f"\n=== PROFIL DE TOXICITÉ : {nom_molecule} ===")
    preds = predictions_medocs[i]
    for index, nom_cible in enumerate(tasks):
        probabilite = preds[index][1]
        if probabilite > 0.5:
            print(f"️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
        else:
            print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")


=== PROFIL DE TOXICITÉ : Paracétamol ===
️ DANGER sur NR-AR      : 64.4%
  Sûr    sur NR-AR-LBD  : 32.0%
️ DANGER sur NR-AhR     : 62.8%
  Sûr    sur NR-Aromatase : 20.6%
️ DANGER sur NR-ER      : 80.8%
️ DANGER sur NR-ER-LBD  : 71.0%
  Sûr    sur NR-PPAR-gamma : 20.9%
️ DANGER sur SR-ARE     : 71.0%
️ DANGER sur SR-ATAD5   : 86.1%
  Sûr    sur SR-HSE     : 46.8%
️ DANGER sur SR-MMP     : 75.0%
️ DANGER sur SR-p53     : 69.0%

=== PROFIL DE TOXICITÉ : Valdécoxib ===
  Sûr    sur NR-AR      : 17.6%
  Sûr    sur NR-AR-LBD  : 30.2%
️ DANGER sur NR-AhR     : 86.0%
  Sûr    sur NR-Aromatase : 41.6%
️ DANGER sur NR-ER      : 75.7%
️ DANGER sur NR-ER-LBD  : 83.6%
️ DANGER sur NR-PPAR-gamma : 91.8%
️ DANGER sur SR-ARE     : 66.1%
️ DANGER sur SR-ATAD5   : 50.4%
️ DANGER sur SR-HSE     : 51.9%
️ DANGER sur SR-MMP     : 63.7%
️ DANGER sur SR-p53     : 59.3%


## Partie 3 : Interprétation chimique des profils de toxicité

### Questions :
4. **Le mystère du Cyanure d'Hydrogène (C#N)** : Le cyanure d'hydrogène est un poison foudroyant extrêmement toxique. Pourtant, observez les résultats du modèle : les probabilités de toxicité sont très faibles sur la plupart des 12 cibles de Tox21. Expliquez ce paradoxe. (Astuce : Réfléchissez au mécanisme d'action réel du cyanure dans le corps humain : attaque-t-il l'ADN ou le récepteur aux œstrogènes ? Ou bloque-t-il la chaîne respiratoire mitochondriale d'une autre manière ?).
5. **Profils comparés (Doliprane vs Valdécoxib)** : Observez les profils de toxicité prédits pour le Paracétamol (Doliprane) et le Valdécoxib.
   * Le modèle permet-il d'avoir une vision plus précise, nuancée et "mécanistique" de la sécurité de ces molécules par rapport à la classification binaire simple de ClinTox ?
   * Quelles cibles spécifiques semblent être menacées par ces deux molécules selon les prédictions ?

*(Double-cliquez sur cette cellule pour rédiger vos réponses ci-dessous :)*

* **Réponse 4 :** ...
* **Réponse 5.1 (Vision mécanique) :** ...
* **Réponse 5.2 (Cibles menacées) :** ...


### 🧬 Annexe : Explication des 12 cibles de toxicité (Tox21)
Le dataset Tox21 analyse la toxicité selon 12 voies biologiques précises. Elles sont divisées en deux grandes catégories :

#### 1. Récepteurs Nucléaires (NR - Perturbateurs Endocriniens)
Ces cibles vérifient si la molécule agit comme un perturbateur hormonal en se fixant sur nos récepteurs.

> **💡 Petit rappel de biologie :**
> * **Une Hormone :** C'est un messager chimique naturel de votre corps (ex: la testostérone, l'insuline). Elle voyage dans le sang et agit comme une "clé" qui rentre dans une "serrure" (le récepteur cellulaire) pour déclencher une action (grandir, stocker du sucre, etc.).
> * **Un Perturbateur Endocrinien :** C'est une molécule chimique artificielle qui a une forme tellement proche d'une vraie hormone qu'elle arrive à pirater la serrure de la cellule. Elle peut soit bloquer la serrure (empêchant la vraie hormone d'agir), soit l'activer au mauvais moment. Cela cause des problèmes de croissance, de fertilité, ou des cancers hormonodépendants.

* **`NR-AR` / `NR-AR-LBD`** : Récepteur aux Androgènes (Hormones masculines comme la testostérone). 
* **`NR-ER` / `NR-ER-LBD`** : Récepteur aux Œstrogènes (Hormones féminines). Souvent lié aux problèmes de fertilité.
* **`NR-Aromatase`** : L'enzyme qui convertit la testostérone en œstrogène.
* **`NR-PPAR-gamma`** : Régulateur du métabolisme des graisses et du sucre (souvent lié au diabète).
* **`NR-AhR`** : Récepteur aux Hydrocarbures (Le récepteur qui réagit aux dioxines et aux polluants chimiques).
  > **Quel est le rôle naturel du AhR dans notre corps ?** 
  > Le récepteur AhR n'est pas une hormone classique. C'est le **"détecteur de poison" naturel** de nos cellules. Son vrai rôle (forgé par l'évolution) est de surveiller notre environnement. S'il détecte des toxines naturelles (par exemple après avoir mangé une plante toxique), il s'active et ordonne à la cellule de produire des enzymes pour détruire et "nettoyer" le poison. 
  > **Le problème moderne :** Les polluants industriels artificiels (comme la dioxine, les pesticides, ou les fumées de diesel) ont des formes géométriques parfaites pour se coincer dans ce récepteur de façon permanente. Le détecteur panique et reste bloqué sur "ON". Le système se dérègle complètement, déclenchant une inflammation chronique massive et de très nombreux cancers.

#### 2. Réponses au Stress Cellulaire (SR - Dégâts et Cancers)
Ces cibles vérifient si la molécule attaque littéralement la machinerie interne de la cellule :
* **`SR-p53`** : La protéine p53 est le "Gardien du Génome". Si elle s'active, cela signifie que la molécule a causé des **dégâts majeurs à l'ADN** (risque très élevé de cancer).
* **`SR-ATAD5`** : Un autre marqueur majeur de génotoxicité (mutations de l'ADN).
* **`SR-MMP`** : Potentiel de la Membrane Mitochondriale. Si la molécule perturbe cela, elle détruit l'"usine à énergie" de la cellule et provoque sa mort (apoptose).
* **`SR-ARE`** : Réponse au stress oxydatif (les fameux radicaux libres).
* **`SR-HSE`** : Réponse aux chocs thermiques et aux protéines mal formées.